# 04 - Embedding baseline (logistic regression)
Trained on train, tuned on validation only. The test set is scored but its metrics are **not** looked at here; that happens once, in notebook 05.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from paths import *
print("repo root:", ROOT)

repo root: C:\Users\yassi\OneDrive\Documents\GitHub\WhoWroteIt-


In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

meta = {s: pd.read_parquet(PROCESSED / f"{s}.parquet") for s in ["train", "val", "test"]}
ID_COL = detect_id_col(meta["train"])
X = {s: np.load(FEATURES / f"emb_{s}.npy") for s in meta}
has = {s: np.load(FEATURES / f"emb_{s}_has.npy") for s in meta}
y = {s: to_binary_label(meta[s][LABEL]) for s in meta}
for s in meta:
    print(s, "rows with embedding:", int(has[s].sum()), "of", len(has[s]))

train rows with embedding: 4670 of 4670
val rows with embedding: 1005 of 1005
test rows with embedding: 1009 of 1009


In [3]:
results = []
best = None
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(C=C, class_weight="balanced", max_iter=3000, random_state=SEED))
    model.fit(X["train"][has["train"]], y["train"][has["train"]])
    s_val = model.predict_proba(X["val"][has["val"]])[:, 1]
    yv = y["val"][has["val"]]
    auc = roc_auc_score(yv, s_val)
    f1 = f1_score(yv, (s_val >= 0.5).astype(int))
    results.append({"C": C, "val_auc": round(auc, 4), "val_f1@0.5": round(f1, 4)})
    if best is None or auc > best[0]:
        best = (auc, C, model)
tuning = pd.DataFrame(results)
display(tuning)
print("chosen C (best validation AUC):", best[1])
model = best[2]

,C,val_auc,val_f1@0.5
0,0.001,0.8108,0.6561
1,0.010,0.8168,0.6368
2,0.100,0.8011,0.6366
3,1.000,0.7762,0.5963
4,10.000,0.7580,0.5775


chosen C (best validation AUC): 0.01


In [4]:
frames = []
for split in ["val", "test"]:
    m = has[split]
    score = model.predict_proba(X[split][m])[:, 1]
    frames.append(pd.DataFrame({
        "sample_id": meta[split][ID_COL].to_numpy()[m],
        "split": split,
        "y_true": y[split][m],
        "y_score": score,
        "y_pred": (score >= 0.5).astype(int),
    }))
preds = pd.concat(frames, ignore_index=True)
RESULTS.mkdir(exist_ok=True)
preds.to_parquet(RESULTS / "preds_embeddings_lr.parquet", index=False)
MODELS.mkdir(exist_ok=True)
joblib.dump(model, MODELS / "embeddings_lr.joblib")
print("saved results/preds_embeddings_lr.parquet:", preds.shape)
print(preds.groupby("split").size())
tuning.to_csv(RESULTS / "final" / "embedding_tuning.csv", index=False) if (RESULTS / "final").exists() else None

saved results/preds_embeddings_lr.parquet: (2014, 5)
split
test    1009
val     1005
dtype: int64
